# Homogeneización de magnitudes a Mw


**TFM — Estimación de zonas de afectación sísmica | Máster Big Data y Visual Analytics — UNIR**

---
### Objetivo
Convertir todas las magnitudes del catálogo sísmico unificado (IG-EPN + USGS) a la escala de magnitud de momento **Mw**, empleando relaciones empíricas de conversión diferenciadas por tipo de magnitud original (`ML`, `Mb`, `Ms`, `MLv`, `Md`, etc.).




---
## Sección 1 — Importaciones y configuración


In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# ── Rutas ─────────────────────────────────────────────────────────
CSV_ENTRADA = r'E:\MASTER_BIG_DATA\TFM_FINAL\DATA\CATALOGOS\Catalogo_Unificado_Ecuador.csv'
CSV_SALIDA  = r'E:\MASTER_BIG_DATA\TFM_2\DATA\DATOS\Catalogo_Sismico_Ecuador_Unificado_Mw.csv'

# ── Parámetros ────────────────────────────────────────────────────
MW_MIN = 4.0   # umbral de magnitud de completitud (Mc = 4.0 Mw)

print('Configuración cargada')
print(f'  Entrada : {CSV_ENTRADA}')
print(f'  Salida  : {CSV_SALIDA}')
print(f'  Umbral  : Mw ≥ {MW_MIN}')


Configuración cargada
  Entrada : E:\MASTER_BIG_DATA\TFM_FINAL\DATA\CATALOGOS\Catalogo_Unificado_Ecuador.csv
  Salida  : E:\MASTER_BIG_DATA\TFM_2\DATA\DATOS\Catalogo_Sismico_Ecuador_Unificado_Mw.csv
  Umbral  : Mw ≥ 4.0


---
## Sección 2 — Carga y diagnóstico del catálogo unificado


In [34]:
df = pd.read_csv(CSV_ENTRADA, sep=',', low_memory=False)
print(f'Catálogo cargado: {df.shape[0]:,} registros × {df.shape[1]} columnas')
print(f'\nColumnas: {list(df.columns)}')
print(f'\nPrimeras filas:')
df.head()


Catálogo cargado: 15,003 registros × 11 columnas

Columnas: ['mag', 'tipo_mag', 'lat', 'lon', 'depth', 'year', 'month', 'day', 'hour', 'minute', 'second']

Primeras filas:


,mag,tipo_mag,lat,lon,depth,year,month,day,hour,minute,second
0,6.40,Mw,0.05,-78.33,10.0,1587.0,8.0,31.0,1.0,30.0,0.0
1,7.00,Mw,-1.73,-78.80,10.0,1645.0,3.0,15.0,0.0,0.0,0.0
2,6.40,Mw,-1.67,-79.05,10.0,1674.0,8.0,29.0,0.0,0.0,0.0
3,6.20,Mw,-1.25,-78.42,10.0,1687.0,11.0,22.0,0.0,0.0,0.0
4,7.25,Mw,-1.65,-78.80,10.0,1698.0,6.0,20.0,6.0,0.0,0.0


In [37]:
# Diagnóstico de tipos de magnitud originales
print('Distribución de tipos de magnitud originales:')
if 'tipo_mag' in df.columns:
    tipo_counts = df['tipo_mag'].value_counts(dropna=False)
    for tipo, cnt in tipo_counts.items():
        print(f'  {str(tipo):<15} {cnt:>6,} registros ({cnt/len(df)*100:.1f}%)')
else:
    print('  Columna tipo_mag no encontrada — se asume todo como magnitud genérica')

print(f'\nRango de magnitud original:')
print(f'  min = {df["mag"].min():.2f} | max = {df["mag"].max():.2f} | media = {df["mag"].mean():.2f}')
print(f'  Nulos en mag: {df["mag"].isna().sum():,}')



Distribución de tipos de magnitud originales:
  Mw              11,635 registros (77.6%)
  M                1,951 registros (13.0%)
  MLv              1,413 registros (9.4%)
  mb                   1 registros (0.0%)
  Mjma                 1 registros (0.0%)
  Mwp                  1 registros (0.0%)
  Mw(Mwp)              1 registros (0.0%)

Rango de magnitud original:
  min = 3.00 | max = 8.35 | media = 3.78
  Nulos en mag: 0


---
## Sección 3 — Funciones de conversión a Mw

Las relaciones empíricas aplicadas fueron calibradas específicamente para el rango de magnitudes de Ecuador (3,0–8,8 Mw) y validadas en la literatura de sismología regional.


In [38]:
def convertir_a_mw(mag, tipo_mag):
    """
    Convierte una magnitud de cualquier tipo a Mw.

    Relaciones empleadas:
      ML  / MLv : Mw = 0.794·ML  + 1.134  (Grünthal et al., 2009)
      Mb        : Mw = 1.084·Mb  - 0.142  (Scordilis, 2006)
      Ms        : Mw = 0.644·Ms  + 2.369  (Scordilis, 2006)
      Md        : Mw = 0.890·Md  + 0.540  (Woessner & Wiemer, 2005)
      Mw y vars : sin conversión
      Otro      : sin conversión (retorna valor original)

    Parámetros
    ----------
    mag      : float  — magnitud original
    tipo_mag : str    — tipo de magnitud (ML, Mb, Ms, Mw, etc.)

    Retorna
    -------
    tuple (mw_calculado: float, convertido: bool)
    """
    if pd.isna(mag):
        return np.nan, False

    tipo = str(tipo_mag).strip().lower() if not pd.isna(tipo_mag) else ''

    # ── Ya está en Mw — sin conversión ────────────────────────────
    if tipo in ['mw', 'mww', 'mwb', 'mwp', 'mwr', 'mwc', 'mwo', 'mwd', 'mwl']:
        return float(mag), False

    # ── ML / MLv — Grünthal et al. (2009) ────────────────────────
    elif tipo in ['ml', 'mlv', 'ml_v', 'ml v', 'local']:
        return 0.794 * float(mag) + 1.134, True

    # ── Mb — Scordilis (2006) ─────────────────────────────────────
    elif tipo in ['mb', 'mb1', 'mb_lg', 'mb1lg']:
        return 1.084 * float(mag) - 0.142, True

    # ── Ms — Scordilis (2006) ─────────────────────────────────────
    elif tipo in ['ms', 'ms_20', 'ms20', 'msa', 'msk']:
        return 0.644 * float(mag) + 2.369, True

    # ── Md — Woessner & Wiemer (2005) ────────────────────────────
    elif tipo in ['md', 'md1', 'duration', 'coda']:
        return 0.890 * float(mag) + 0.540, True

    # ── Tipo desconocido — sin conversión ─────────────────────────
    else:
        return float(mag), False


# Vectorizar para aplicar sobre el DataFrame
convertir_v = np.vectorize(convertir_a_mw, otypes=[float, bool])

print('Funciones de conversión definidas:')
tests = [('ML',  5.0), ('MLv', 4.5), ('Mb',  5.5), ('Ms', 6.0),
          ('Md',  4.0), ('Mw',  6.5), ('mww', 7.0), ('?',  5.0)]
print(f'  {"Tipo":<8} {"Original":>10} {"→ Mw":>10} {"Convertido":>12}')
print(f'  {"-"*42}')
for tipo, val in tests:
    mw, conv = convertir_a_mw(val, tipo)
    print(f'  {tipo:<8} {val:>10.2f} {mw:>10.4f} {str(conv):>12}')


Funciones de conversión definidas:
  Tipo       Original       → Mw   Convertido
  ------------------------------------------
  ML             5.00     5.1040         True
  MLv            4.50     4.7070         True
  Mb             5.50     5.8200         True
  Ms             6.00     6.2330         True
  Md             4.00     4.1000         True
  Mw             6.50     6.5000        False
  mww            7.00     7.0000        False
  ?              5.00     5.0000        False


---
## Sección 4 — Aplicación de las relaciones de conversión


In [ ]:
# Asegurar que tipo_mag existe
if 'tipo_mag' not in df.columns:
    df['tipo_mag'] = 'mw'   # si no hay columna, asumir Mw
df['tipo_mag'] = df['tipo_mag'].fillna('desconocido').astype(str).str.strip()

# Aplicar conversión vectorizada
print('Aplicando conversiones...')
mw_vals, conv_flags = convertir_v(df['mag'].values, df['tipo_mag'].values)

df['mag_original']  = df['mag'].copy()          # guardar valor original
df['mag']           = mw_vals                    # reemplazar con Mw
df['mag_convertida'] = conv_flags                # bandera de conversión

# Estadísticas de conversión
n_total     = len(df)
n_convert   = df['mag_convertida'].sum()
n_ya_mw     = n_total - n_convert

print(f'\nResumen de conversión:')
print(f'  Total registros          : {n_total:,}')
print(f'  Ya en Mw (sin cambio)    : {n_ya_mw:,} ({n_ya_mw/n_total*100:.1f}%)')
print(f'  Convertidos a Mw         : {n_convert:,} ({n_convert/n_total*100:.1f}%)')
print(f'\nDetalle por tipo de magnitud:')
resumen = df.groupby('tipo_mag').agg(
    n_registros=('mag','count'),
    mag_orig_media=('mag_original','mean'),
    mw_media=('mag','mean'),
    convertidos=('mag_convertida','sum')
).sort_values('n_registros', ascending=False)
print(resumen.to_string())


---
## Sección 5 — Filtros de depuración y control de calidad


In [ ]:
n_antes = len(df)
print(f'Registros antes de depuración: {n_antes:,}')

# 1. Eliminar nulos en campos críticos
df.dropna(subset=['mag','lat','lon','depth'], inplace=True)
print(f'  Tras eliminar nulos críticos : {len(df):,} (−{n_antes-len(df):,})')

# 2. Aplicar umbral de magnitud de completitud (Mc = 3.0 Mw)
n_pre = len(df)
df    = df[df['mag'] >= MW_MIN].copy()
print(f'  Tras filtro Mw ≥ {MW_MIN}         : {len(df):,} (−{n_pre-len(df):,})')

# 3. Rango geográfico Ecuador continental
n_pre = len(df)
df    = df[(df['lat'].between(-5.5, 1.5)) &
           (df['lon'].between(-81.5, -75.0))].copy()
print(f'  Tras filtro geográfico       : {len(df):,} (−{n_pre-len(df):,})')

# 4. Profundidad máxima razonable (700 km)
n_pre = len(df)
df    = df[df['depth'] <= 700].copy()
print(f'  Tras filtro profundidad ≤700 : {len(df):,} (−{n_pre-len(df):,})')

# 5. Magnitud máxima razonable (10 Mw)
n_pre = len(df)
df    = df[df['mag'] <= 10.0].copy()
print(f'  Tras filtro mag ≤ 10         : {len(df):,} (−{n_pre-len(df):,})')

# 6. Eliminar duplicados (mismo tiempo + coordenadas)
dup_cols = ['year','month','day','hour','lat','lon']
dup_cols = [c for c in dup_cols if c in df.columns]
if dup_cols:
    n_pre = len(df)
    df.drop_duplicates(subset=dup_cols, keep='first', inplace=True)
    print(f'  Tras eliminar duplicados     : {len(df):,} (−{n_pre-len(df):,})')

# 7. Ordenar cronológicamente
sort_cols = [c for c in ['year','month','day','hour','minute','second'] if c in df.columns]
df.sort_values(sort_cols, inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'\nCatálogo depurado final: {len(df):,} registros')
print(f'  Mw: [{df["mag"].min():.2f}, {df["mag"].max():.2f}] | media={df["mag"].mean():.2f}')


---
## Sección 6 — Visualización: antes y después de la conversión


In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

# Distribución magnitudes antes vs después
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df['mag_original'], bins=40, color='#F44336', edgecolor='white',
         alpha=0.75, label='Magnitud original')
ax1.hist(df['mag'], bins=40, color='#2196F3', edgecolor='white',
         alpha=0.60, label='Mw convertida')
ax1.axvline(MW_MIN, color='orange', ls='--', lw=1.5, label=f'Mc={MW_MIN}')
ax1.set_title('Distribución magnitudes\nantes vs después de conversión',
               fontweight='bold')
ax1.set_xlabel('Magnitud'); ax1.set_ylabel('Frecuencia')
ax1.legend(fontsize=8)

# Scatter magnitud original vs Mw resultante
ax2 = fig.add_subplot(gs[0, 1])
mask_c = df['mag_convertida']
ax2.scatter(df.loc[~mask_c,'mag_original'], df.loc[~mask_c,'mag'],
            s=4, alpha=0.3, color='#9E9E9E', label='Sin conversión')
ax2.scatter(df.loc[mask_c,'mag_original'],  df.loc[mask_c,'mag'],
            s=4, alpha=0.5, color='#D32F2F', label='Convertida')
mn = min(df['mag_original'].min(), df['mag'].min())
mx = max(df['mag_original'].max(), df['mag'].max())
ax2.plot([mn,mx],[mn,mx],'k--',lw=1,label='y=x (sin cambio)')
ax2.set_title('Magnitud original vs Mw resultante', fontweight='bold')
ax2.set_xlabel('Magnitud original'); ax2.set_ylabel('Mw')
ax2.legend(fontsize=7)

# Pie de tipos de magnitud
ax3 = fig.add_subplot(gs[0, 2])
tipo_cnt = df['tipo_mag'].value_counts().head(8)
ax3.pie(tipo_cnt.values, labels=tipo_cnt.index, autopct='%1.1f%%',
        startangle=90, textprops={'fontsize':8})
ax3.set_title('Distribución tipos\nde magnitud original', fontweight='bold')

# Mapa epicentros
ax4 = fig.add_subplot(gs[1, 0:2])
sc = ax4.scatter(df['lon'], df['lat'], c=df['mag'], cmap='plasma',
                  s=3, alpha=0.4, vmin=MW_MIN, vmax=8.0)
plt.colorbar(sc, ax=ax4, label='Mw')
ax4.set_title('Distribución espacial del catálogo depurado', fontweight='bold')
ax4.set_xlabel('Longitud'); ax4.set_ylabel('Latitud')

# Histograma Gutenberg-Richter
ax5 = fig.add_subplot(gs[1, 2])
bins = np.arange(MW_MIN, df['mag'].max()+0.5, 0.5)
counts, edges = np.histogram(df['mag'], bins=bins)
cum_counts = counts[::-1].cumsum()[::-1]
ax5.semilogy(edges[:-1], cum_counts, 'o-', color='#1565C0', ms=5, lw=1.5)
ax5.axvline(MW_MIN, color='orange', ls='--', lw=1.5, label=f'Mc={MW_MIN}')
ax5.set_title('Relación Gutenberg-Richter\n(distribución acumulada)', fontweight='bold')
ax5.set_xlabel('Mw'); ax5.set_ylabel('log N (≥ Mw)')
ax5.legend(fontsize=9); ax5.grid(True, alpha=0.3)

plt.suptitle(f'Diagnóstico del catálogo sísmico unificado Ecuador\n'
              f'N = {len(df):,} eventos | Mw ≥ {MW_MIN} | 1587–2025',
              fontsize=13, fontweight='bold')
plt.show()


---
## Sección 7 — Exportación del catálogo homogeneizado


In [ ]:
# Columnas finales del catálogo
COLS_ORDEN = ['mag','tipo_mag','mag_original','mag_convertida',
               'lat','lon','depth',
               'year','month','day','hour','minute','second']
COLS_FINAL = [c for c in COLS_ORDEN if c in df.columns]
# Agregar columnas adicionales que puedan existir
extra = [c for c in df.columns if c not in COLS_FINAL]
COLS_FINAL += extra

df[COLS_FINAL].to_csv(CSV_SALIDA, index=False, encoding='utf-8')

print(f'Catálogo exportado: {CSV_SALIDA}')
print(f'  Registros   : {len(df):,}')
print(f'  Columnas    : {len(COLS_FINAL)}')
print(f'  Mw mínima   : {df["mag"].min():.2f}')
print(f'  Mw máxima   : {df["mag"].max():.2f}')
print(f'  Período     : {int(df["year"].min())} – {int(df["year"].max())}')
print(f'  Convertidos : {df["mag_convertida"].sum():,} ({df["mag_convertida"].mean()*100:.1f}%)')
print(f'\nListo.')
